# Beca 18 RAG Chatbot — Document Retrieval and Grounded Generation

**Course:** Python Programming — Applied Data Science  
**Source:** Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC  

Pipeline: `PDF → text extraction → chunking → embeddings → ChromaDB → retrieval → grounded generation`

## Step 0 — Setup: Dependencies and Environment

In [ ]:
!pip install pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv -q

import pypdf
import tiktoken
import langchain
import google.genai as genai_pkg
import chromadb
import ipywidgets
import tqdm as tqdm_module
import dotenv

def get_ver(pkg):
    return getattr(pkg, '__version__', 'unknown')

print(f"pypdf:          {get_ver(pypdf)}")
print(f"tiktoken:       {get_ver(tiktoken)}")
print(f"langchain:      {get_ver(langchain)}")
print(f"google-genai:   {get_ver(genai_pkg)}")
print(f"chromadb:       {get_ver(chromadb)}")
print(f"ipywidgets:     {get_ver(ipywidgets)}")
print(f"tqdm:           {get_ver(tqdm_module)}")
print(f"python-dotenv:  {get_ver(dotenv)}")
print("\nEnvironment ready.")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Create a .env file with: GEMINI_API_KEY=your_key_here")
print("GEMINI_API_KEY loaded successfully.")

from google import genai as google_genai
client = google_genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized.")

## Step 1 — PDF Text Extraction

In [ ]:
import pypdf
import re

pdf_path = "../data/beca18_reglamento.pdf"

def extract_text_from_pdf(path):
    """Extract text page by page with [PAGE N] markers and light cleaning."""
    full_text = ""
    text_pages = []
    with open(path, "rb") as f:
        reader = pypdf.PdfReader(f)
        total_pages = len(reader.pages)
        print(f"Total pages in PDF: {total_pages}")
        for i, page in enumerate(reader.pages):
            raw = page.extract_text() or ""
            cleaned = re.sub(r"[ \t]+", " ", raw)
            cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
            cleaned = cleaned.strip()
            block = f"[PAGE {i+1}]\n{cleaned}"
            text_pages.append(block)
            full_text += block + "\n\n"
    return full_text, text_pages

full_text, text_pages = extract_text_from_pdf(pdf_path)

total_chars = len(full_text)
total_words = len(full_text.split())
print(f"Total characters: {total_chars:,}")
print(f"Total words:      {total_words:,}")
print(f"\n--- Preview (first 600 chars) ---")
print(full_text[:600])

## Step 2 — Tokenization, Chunking Justification, and Chunking

### Chunking Justification

Using `cl100k_base` encoding (compatible with Gemini embedding models), the total token count is computed below.

**Why chunk_size = 400 tokens with 60-token overlap?**

- The `gemini-embedding-001` model supports up to **8,192 tokens** per input.
- A chunk of 400 tokens leaves substantial headroom, ensuring every chunk fits within the embedding limit even after metadata injection.
- 400 tokens ≈ 300–350 words — enough to capture a complete regulatory clause without fragmenting key concepts.
- A 60-token overlap (15%) preserves cross-chunk continuity: sentences straddling chunk boundaries are still represented in at least one complete chunk, reducing retrieval information loss.
- Smaller chunks (< 200 tokens) produce too many noisy fragments; larger chunks (> 800 tokens) risk semantic dilution that hurts retrieval precision.

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Count tokens
encoding     = tiktoken.get_encoding("cl100k_base")
tokens       = encoding.encode(full_text)
total_tokens = len(tokens)
print(f"Total tokens (cl100k_base): {total_tokens:,}")
print(f"Embedding model limit:       8,192 tokens")
print(f"Chosen chunk_size:           400 tokens")
print(f"Chosen chunk_overlap:         60 tokens (15% overlap)")

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "],
)

doc_metadata = {
    "document": "Resolucion Directoral Ejecutiva N.033-2026-MINEDU/VMGI-PRONABEC",
    "topic":    "Beca 18 Regulations",
    "language": "Spanish",
}

chunks = splitter.create_documents([full_text], metadatas=[doc_metadata])

total_chunks = len(chunks)
avg_len      = sum(len(c.page_content) for c in chunks) / total_chunks if chunks else 0
print(f"\nTotal chunks:     {total_chunks}")
print(f"Avg chunk length: {avg_len:.0f} characters")
print(f"\n--- Example chunk (first) ---")
print(chunks[0].page_content[:400])

## Step 3 — Embeddings with Exponential Backoff

In [ ]:
import time

EMBED_MODEL = "gemini-embedding-001"

def embed_with_backoff(texts, task_type, max_retries=6):
    """Embed texts using Gemini with exponential backoff for free-tier rate limits (~60 req/min)."""
    from google.genai import types as genai_types
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model=EMBED_MODEL,
                contents=texts,
                config=genai_types.EmbedContentConfig(task_type=task_type),
            )
            return [e.values for e in result.embeddings]
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            wait = (2 ** attempt) + (0.5 * attempt)
            print(f"  [attempt {attempt+1}] Rate limit: {exc} — retrying in {wait:.1f}s")
            time.sleep(wait)

def embed_documents(texts):
    """Embed documents for indexing (RETRIEVAL_DOCUMENT task type)."""
    return embed_with_backoff(texts, "RETRIEVAL_DOCUMENT")

def embed_query(text: str):
    """Embed a single query string for search (RETRIEVAL_QUERY task type)."""
    return embed_with_backoff([text], "RETRIEVAL_QUERY")[0]

# Smoke test
test_vec = embed_query("Que es la Beca 18?")
print(f"Embedding dimensions: {len(test_vec)}")
print("Embedding functions ready.")

## Step 4 — Vector Database (ChromaDB, Persistent, Cosine Distance)

In [ ]:
import chromadb
from tqdm.notebook import tqdm
import time

CHROMA_PATH     = "chroma_db_beca18"
COLLECTION_NAME = "beca18_regulations"
BATCH_SIZE      = 8

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Idempotent indexing: skip if collection is already populated
existing_names = [c.name for c in chroma_client.list_collections()]

if COLLECTION_NAME in existing_names:
    collection     = chroma_client.get_collection(COLLECTION_NAME)
    existing_count = collection.count()
    print(f"Collection '{COLLECTION_NAME}' found with {existing_count} documents.")
    skip_embedding = existing_count > 0
else:
    collection     = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
    skip_embedding = False
    print(f"Created new collection: '{COLLECTION_NAME}'")

if skip_embedding:
    print("Skipping embedding step — collection already populated.")
else:
    print(f"Embedding {len(chunks)} chunks in batches of {BATCH_SIZE}...")
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch  = chunks[i : i + BATCH_SIZE]
        texts  = [c.page_content for c in batch]
        metas  = [c.metadata     for c in batch]
        ids    = [f"chunk_{i+j}" for j in range(len(batch))]
        embeds = embed_documents(texts)
        collection.add(embeddings=embeds, documents=texts, metadatas=metas, ids=ids)
        if i + BATCH_SIZE < len(chunks):
            time.sleep(1.5)

print(f"\nTotal documents stored: {collection.count()}")

## Step 5 — Semantic Search

In [ ]:
def semantic_search(question: str, k: int = 5):
    """Embed the question and retrieve top-k nearest chunks from ChromaDB.
    Returns a list of dicts with keys: text, metadata, distance.
    """
    q_vec   = embed_query(question)
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    output  = []
    for idx in range(len(results["documents"][0])):
        output.append({
            "text":     results["documents"][0][idx],
            "metadata": results["metadatas"][0][idx],
            "distance": results["distances"][0][idx],
        })
    return output

# --- Test with one sample question ---
sample_q = "Cuales son los requisitos para postular a la Beca 18?"
print(f"Query: {sample_q}\n")
top_results = semantic_search(sample_q, k=3)
for i, r in enumerate(top_results, 1):
    print(f"[Result {i}]  distance={r['distance']:.4f}")
    print(f"  {r['text'][:250]}")
    print()

## Step 6 — Grounded Generation with Gemini 2.5 Flash

In [ ]:
from google.genai import types as genai_types

GENERATION_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = (
    "Eres un asistente especializado en el reglamento de la Beca 18 (PRONABEC, Peru).\n"
    "Tu funcion es responder preguntas EXCLUSIVAMENTE basandote en el contexto documentado que se te proporciona.\n\n"
    "Reglas obligatorias:\n"
    "1. Responde SOLO con informacion del contexto proporcionado. Nunca uses conocimiento externo ni hagas suposiciones.\n"
    "2. Cita el numero de pagina cuando este disponible en el fragmento (ej: [ver pagina 5]).\n"
    "3. Si el contexto no contiene informacion suficiente para responder, responde exactamente: "
    "'El documento no contiene informacion sobre este tema.'\n"
    "4. Se preciso, claro y conciso. Sintetiza coherentemente si varios fragmentos son relevantes.\n"
    "5. No inventes datos como montos, fechas o requisitos que no aparezcan en el contexto."
)

def answer_with_context(question: str, k: int = 5):
    """Retrieve relevant chunks and generate a grounded answer using Gemini 2.5 Flash."""
    retrieved = semantic_search(question, k=k)
    parts = []
    for i, r in enumerate(retrieved, 1):
        parts.append(f"[Fragmento {i} | distancia={r['distance']:.3f}]\n{r['text']}")
    context_str = "\n\n".join(parts)

    user_msg = (
        f"Contexto recuperado del documento oficial:\n\n{context_str}\n\n"
        f"Pregunta: {question}\n\n"
        "Responde basandote unicamente en el contexto anterior."
    )

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=user_msg,
        config=genai_types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
        ),
    )
    return response.text, retrieved

# --- 5 on-topic questions ---
on_topic = [
    "Cuales son los requisitos de elegibilidad para postular a la Beca 18?",
    "Que modalidades de becas ofrece el programa Beca 18?",
    "Cual es el monto del estipendio o subvencion mensual que recibe el becario?",
    "Cuales son las obligaciones del becario durante el periodo de estudios?",
    "En que condiciones o causales se pierde la Beca 18?",
]

for q in on_topic:
    print(f"{'='*65}")
    print(f"Q: {q}")
    ans, _ = answer_with_context(q)
    print(f"A: {ans}\n")

# --- 1 off-topic question ---
off_q = "Cual es la capital de Francia y cuantos habitantes tiene Paris?"
print(f"{'='*65}")
print(f"Q (off-topic): {off_q}")
ans_off, _ = answer_with_context(off_q)
print(f"A: {ans_off}")

## Step 7 — Interactive Chat Interface (ipywidgets)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import re

# --- UI Components ---
header = widgets.HTML(
    "<h3 style='color:#1a237e;margin-bottom:4px;'>Beca 18 RAG Chatbot</h3>"
    "<p style='color:#555;margin-top:0;'>Powered by Gemini 2.5 Flash + ChromaDB</p>"
)
q_input = widgets.Textarea(
    placeholder="Escribe tu pregunta sobre el reglamento de la Beca 18...",
    layout=widgets.Layout(width="85%", height="80px"),
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k (fragmentos):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="55%"),
)
ask_btn   = widgets.Button(description="Preguntar", button_style="primary", icon="search")
clear_btn = widgets.Button(description="Limpiar",   button_style="warning", icon="trash")
ans_out   = widgets.Output(layout=widgets.Layout(
    border="1px solid #90CAF9", padding="12px", min_height="80px"
))
src_out_init   = widgets.Output()
sources_acc    = widgets.Accordion(children=[src_out_init])
sources_acc.set_title(0, "Fragmentos recuperados (click para expandir)")

def on_ask(b):
    question = q_input.value.strip()
    if not question:
        return
    with ans_out:
        clear_output(wait=True)
        print("Buscando respuesta...")
    try:
        answer, retrieved = answer_with_context(question, k=k_slider.value)
        with ans_out:
            clear_output(wait=True)
            display(HTML(f"<b>Pregunta:</b> {question}<br><br><b>Respuesta:</b><br>{answer.replace(chr(10), '<br>')}"))
        new_src = widgets.Output()
        with new_src:
            for i, r in enumerate(retrieved, 1):
                page_tag = ""
                m = re.search(r"\[PAGE (\d+)\]", r["text"])
                if m:
                    page_tag = f" | Pagina {m.group(1)}"
                print(f"--- Fragmento {i}{page_tag} | distancia={r['distance']:.4f} ---")
                print(r["text"][:400])
                print()
        sources_acc.children = [new_src]
        sources_acc.set_title(0, f"Fragmentos recuperados ({len(retrieved)}) — click para expandir")
    except Exception as e:
        with ans_out:
            clear_output(wait=True)
            print(f"Error: {e}")

def on_clear(b):
    q_input.value = ""
    with ans_out:
        clear_output()
    empty = widgets.Output()
    sources_acc.children = [empty]
    sources_acc.set_title(0, "Fragmentos recuperados (click para expandir)")

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

ui = widgets.VBox([
    header,
    q_input,
    k_slider,
    widgets.HBox([ask_btn, clear_btn]),
    ans_out,
    sources_acc,
])
display(ui)